
# Install required R packages
install.packages(c(
  "tidyverse",
  "forecast", 
  "randomForest",
  "caret",
  "lubridate",
  "plotly",
  "DT",
  "zoo",
  "here"
), repos = "https://cloud.r-project.org")

cat("✓ All packages installed successfully!\n")


# Load required libraries
library(tidyverse)
library(forecast)
library(randomForest)
library(caret)
library(lubridate)
library(plotly)
library(DT)
library(zoo)
library(here)

cat("✓ All libraries loaded successfully!\n")

In [ ]:

# Create sample GSE stock data (since we don't have the actual data files in Colab)
set.seed(123)
dates <- seq(as.Date("2023-01-01"), as.Date("2023-12-31"), by = "day")
stocks <- c("ACCESS", "ADB", "AGA", "ASG", "ALLGH", "BOPP", "CAL", "CLYD", "CMLT", "EGH")

# Generate sample data
sample_data <- expand.grid(
  date = dates,
  share_code = stocks
) %>%
  mutate(
    closing_price_vwap = runif(n(), 1, 100),
    total_shares_traded = runif(n(), 1000, 100000),
    opening_price = closing_price_vwap * runif(n(), 0.95, 1.05),
    price_change = closing_price_vwap - lag(closing_price_vwap, default = closing_price_vwap[1])
  ) %>%
  arrange(share_code, date)

cat("✓ Sample data created with", nrow(sample_data), "rows\n")
print(head(sample_data))

In [ ]:

# Technical indicator functions
calculate_rsi <- function(prices, period = 14) {
  if (length(prices) < period + 1) return(rep(NA, length(prices)))
  
  gains <- pmax(diff(prices), 0)
  losses <- pmax(-diff(prices), 0)
  
  avg_gain <- zoo::rollmean(gains, period, fill = NA, align = "right")
  avg_loss <- zoo::rollmean(losses, period, fill = NA, align = "right")
  
  rs <- avg_gain / avg_loss
  rsi <- 100 - (100 / (1 + rs))
  
  return(c(rep(NA, 1), rsi))
}

calculate_macd <- function(prices, fast = 12, slow = 26) {
  if (length(prices) < slow) return(rep(NA, length(prices)))
  
  ema_fast <- zoo::rollmean(prices, fast, fill = NA, align = "right")
  ema_slow <- zoo::rollmean(prices, slow, fill = NA, align = "right")
  
  macd_line <- ema_fast - ema_slow
  return(macd_line)
}

calculate_bollinger_bands <- function(prices, period = 20, std_dev = 2) {
  if (length(prices) < period) return(list(upper = rep(NA, length(prices)), lower = rep(NA, length(prices))))
  
  sma <- zoo::rollmean(prices, period, fill = NA, align = "right")
  std <- zoo::rollapply(prices, period, sd, fill = NA, align = "right")
  
  list(
    upper = sma + (std * std_dev),
    lower = sma - (std * std_dev)
  )
}

cat("✓ Technical indicator functions defined\n")

In [ ]:
# Function to prepare data for modeling
prepare_modeling_data <- function(data, target_stock = NULL) {
  if (is.null(data)) return(NULL)
  
  cat("Preparing data for modeling...\n")
  
  # Filter for specific stock if provided
  if (!is.null(target_stock)) {
    data <- data %>%
      filter(share_code == target_stock)
    cat(paste("Filtered for stock:", target_stock, "\n"))
  }
  
  # Ensure data is sorted by date
  data <- data %>%
    arrange(date) %>%
    filter(!is.na(closing_price_vwap)) %>%
    filter(closing_price_vwap > 0)
  
  # Create lagged features
  data <- data %>%
    group_by(share_code) %>%
    mutate(
      price_lag_1 = lag(closing_price_vwap, 1),
      price_lag_2 = lag(closing_price_vwap, 2),
      price_lag_3 = lag(closing_price_vwap, 3),
      price_lag_5 = lag(closing_price_vwap, 5),
      price_lag_10 = lag(closing_price_vwap, 10),
      
      # Price changes
      price_change_1 = closing_price_vwap - price_lag_1,
      price_change_2 = closing_price_vwap - price_lag_2,
      price_change_5 = closing_price_vwap - price_lag_5,
      
      # Volume features
      volume_lag_1 = lag(total_shares_traded, 1),
      volume_change = total_shares_traded - volume_lag_1,
      
      # Technical indicators
      rsi = calculate_rsi(closing_price_vwap, 14),
      macd = calculate_macd(closing_price_vwap),
      
      # Moving averages
      ma_5 = zoo::rollmean(closing_price_vwap, 5, fill = NA, align = "right"),
      ma_10 = zoo::rollmean(closing_price_vwap, 10, fill = NA, align = "right"),
      ma_20 = zoo::rollmean(closing_price_vwap, 20, fill = NA, align = "right"),
      
      # Volatility
      volatility_5 = zoo::rollapply(closing_price_vwap, 5, sd, fill = NA, align = "right"),
      volatility_10 = zoo::rollapply(closing_price_vwap, 10, sd, fill = NA, align = "right")
    ) %>%
    ungroup()
  
  # Remove rows with missing values
  data <- data %>%
    filter(!is.na(price_lag_1)) %>%
    filter(!is.na(price_lag_2)) %>%
    filter(!is.na(price_lag_3))
  
  cat(paste("Prepared data:", nrow(data), "rows\n"))
  
  return(data)
}

# Prepare the sample data
modeling_data <- prepare_modeling_data(sample_data)
cat("Data preprocessing completed\n")

In [ ]:
# ARIMA Model Function
train_arima_model <- function(data, stock_code) {
  cat(paste("Training ARIMA model for", stock_code, "...\n"))
  
  # Filter data for specific stock
  stock_data <- data %>%
    filter(share_code == stock_code) %>%
    arrange(date)
  
  if (nrow(stock_data) < 30) {
    cat(paste("⚠ Insufficient data for", stock_code, "\n"))
    return(NULL)
  }
  
  # Create time series
  ts_data <- ts(stock_data$closing_price_vwap, frequency = 1)
  
  # Split data into train and test
  train_size <- floor(0.8 * length(ts_data))
  train_data <- ts_data[1:train_size]
  test_data <- ts_data[(train_size + 1):length(ts_data)]
  
  tryCatch({
    # Fit ARIMA model
    arima_model <- auto.arima(train_data,
                              seasonal = FALSE,
                              stepwise = TRUE,
                              approximation = TRUE)
    
    # Make predictions
    forecast_result <- forecast(arima_model, h = length(test_data))
    
    # Calculate metrics
    predictions <- as.numeric(forecast_result$mean)
    actual <- as.numeric(test_data)
    
    mae <- mean(abs(predictions - actual), na.rm = TRUE)
    rmse <- sqrt(mean((predictions - actual)^2, na.rm = TRUE))
    mape <- mean(abs((actual - predictions) / actual), na.rm = TRUE) * 100
    
    # Directional accuracy
    actual_direction <- sign(diff(actual))
    pred_direction <- sign(diff(predictions))
    directional_accuracy <- mean(actual_direction == pred_direction, na.rm = TRUE) * 100
    
    model_results <- list(
      model = arima_model,
      predictions = predictions,
      actual = actual,
      mae = mae,
      rmse = rmse,
      mape = mape,
      directional_accuracy = directional_accuracy,
      stock_code = stock_code,
      model_type = "ARIMA"
    )
    
    cat(paste("✓ ARIMA model trained for", stock_code, "\n"))
    cat(paste("  MAE:", round(mae, 4), "\n"))
    cat(paste("  RMSE:", round(rmse, 4), "\n"))
    cat(paste("  MAPE:", round(mape, 2), "%\n"))
    cat(paste("  Directional Accuracy:", round(directional_accuracy, 2), "%\n"))
    
    return(model_results)
    
  }, error = function(e) {
    cat(paste("✗ Error training ARIMA for", stock_code, ":", e$message, "\n"))
    NULL
  })
}

cat("✓ ARIMA model function defined\n")

In [ ]:
# Random Forest Model Function
train_random_forest_model <- function(data, stock_code) {
  cat(paste("Training Random Forest model for", stock_code, "...\n"))
  
  # Filter data for specific stock
  stock_data <- data %>%
    filter(share_code == stock_code) %>%
    arrange(date) %>%
    filter(!is.na(price_lag_1) & !is.na(price_lag_2) & !is.na(price_lag_3))
  
  if (nrow(stock_data) < 30) {
    cat(paste("⚠ Insufficient data for", stock_code, "\n"))
    return(NULL)
  }
  
  # Prepare features
  features <- c("price_lag_1", "price_lag_2", "price_lag_3", "price_lag_5", 
                "ma_5", "ma_10", "ma_20", "volatility_5", "volatility_10",
                "rsi", "macd")
  
  # Select available features
  available_features <- features[features %in% names(stock_data)]
  
  # Prepare data for Random Forest
  rf_data <- stock_data %>%
    select(all_of(c("closing_price_vwap", available_features))) %>%
    filter(complete.cases(.))
  
  if (nrow(rf_data) < 20) {
    cat(paste("⚠ Insufficient complete cases for", stock_code, "\n"))
    return(NULL)
  }
  
  # Split data
  train_size <- floor(0.8 * nrow(rf_data))
  train_data <- rf_data[1:train_size, ]
  test_data <- rf_data[(train_size + 1):nrow(rf_data), ]
  
  tryCatch({
    # Train Random Forest
    rf_model <- randomForest(closing_price_vwap ~ .,
                             data = train_data,
                             ntree = 100,
                             mtry = floor(sqrt(length(available_features))),
                             importance = TRUE)
    
    # Make predictions
    predictions <- predict(rf_model, newdata = test_data)
    actual <- test_data$closing_price_vwap
    
    # Calculate metrics
    mae <- mean(abs(predictions - actual), na.rm = TRUE)
    rmse <- sqrt(mean((predictions - actual)^2, na.rm = TRUE))
    mape <- mean(abs((actual - predictions) / actual), na.rm = TRUE) * 100
    
    # Directional accuracy
    actual_direction <- sign(diff(actual))
    pred_direction <- sign(diff(predictions))
    directional_accuracy <- mean(actual_direction == pred_direction, na.rm = TRUE) * 100
    
    model_results <- list(
      model = rf_model,
      predictions = predictions,
      actual = actual,
      mae = mae,
      rmse = rmse,
      mape = mape,
      directional_accuracy = directional_accuracy,
      stock_code = stock_code,
      model_type = "Random Forest"
    )
    
    cat(paste("✓ Random Forest model trained for", stock_code, "\n"))
    cat(paste("  MAE:", round(mae, 4), "\n"))
    cat(paste("  RMSE:", round(rmse, 4), "\n"))
    cat(paste("  MAPE:", round(mape, 2), "%\n"))
    cat(paste("  Directional Accuracy:", round(directional_accuracy, 2), "%\n"))
    
    return(model_results)
    
  }, error = function(e) {
    cat(paste("✗ Error training Random Forest for", stock_code, ":", e$message, "\n"))
    NULL
  })
}

cat("✓ Random Forest model function defined\n")

In [ ]:
# Train models for a specific stock
stock_to_predict <- "ACCESS"  # Change this to any stock code

cat(paste("Training models for stock:", stock_to_predict, "\n"))

# Train ARIMA model
arima_result <- train_arima_model(modeling_data, stock_to_predict)

# Train Random Forest model
rf_result <- train_random_forest_model(modeling_data, stock_to_predict)

cat("✓ Model training completed\n")

In [ ]:
# Visualization functions
plot_predictions <- function(model_result, title = NULL) {
  if (is.null(model_result)) return(NULL)
  
  # Create data frame for plotting
  plot_data <- data.frame(
    index = 1:length(model_result$actual),
    actual = model_result$actual,
    predicted = model_result$predictions
  )
  
  # Create plot
  p <- plot_ly(plot_data, x = ~index) %>%
    add_lines(y = ~actual, name = "Actual", line = list(color = "blue")) %>%
    add_lines(y = ~predicted, name = "Predicted", line = list(color = "red")) %>%
    layout(
      title = paste(title, "-", model_result$stock_code, model_result$model_type),
      xaxis = list(title = "Time Index"),
      yaxis = list(title = "Price (GH¢)"),
      hovermode = "x unified"
    )
  
  return(p)
}

plot_model_comparison <- function(arima_result, rf_result) {
  if (is.null(arima_result) && is.null(rf_result)) return(NULL)
  
  # Create comparison data
  comparison_data <- data.frame(
    Model = c(),
    MAE = c(),
    RMSE = c(),
    MAPE = c(),
    Directional_Accuracy = c()
  )
  
  if (!is.null(arima_result)) {
    comparison_data <- rbind(comparison_data, data.frame(
      Model = "ARIMA",
      MAE = arima_result$mae,
      RMSE = arima_result$rmse,
      MAPE = arima_result$mape,
      Directional_Accuracy = arima_result$directional_accuracy
    ))
  }
  
  if (!is.null(rf_result)) {
    comparison_data <- rbind(comparison_data, data.frame(
      Model = "Random Forest",
      MAE = rf_result$mae,
      RMSE = rf_result$rmse,
      MAPE = rf_result$mape,
      Directional_Accuracy = rf_result$directional_accuracy
    ))
  }
  
  # Create bar plot
  p <- plot_ly(comparison_data, x = ~Model, y = ~MAE, type = "bar", name = "MAE") %>%
    add_trace(y = ~RMSE, name = "RMSE") %>%
    add_trace(y = ~MAPE, name = "MAPE") %>%
    add_trace(y = ~Directional_Accuracy, name = "Directional Accuracy") %>%
    layout(
      title = "Model Performance Comparison",
      xaxis = list(title = "Model"),
      yaxis = list(title = "Metric Value"),
      barmode = "group"
    )
  
  return(p)
}

cat("✓ Visualization functions defined\n")

In [ ]:
# Generate predictions plots
if (!is.null(arima_result)) {
  arima_plot <- plot_predictions(arima_result, "ARIMA Predictions")
  print(arima_plot)
}

if (!is.null(rf_result)) {
  rf_plot <- plot_predictions(rf_result, "Random Forest Predictions")
  print(rf_plot)
}

# Generate model comparison
comparison_plot <- plot_model_comparison(arima_result, rf_result)
if (!is.null(comparison_plot)) {
  print(comparison_plot)
}

In [ ]:
# Create results summary
create_results_summary <- function(arima_result, rf_result) {
  summary_data <- data.frame(
    Model = character(),
    Stock_Code = character(),
    MAE = numeric(),
    RMSE = numeric(),
    MAPE = numeric(),
    Directional_Accuracy = numeric()
  )
  
  if (!is.null(arima_result)) {
    summary_data <- rbind(summary_data, data.frame(
      Model = "ARIMA",
      Stock_Code = arima_result$stock_code,
      MAE = round(arima_result$mae, 4),
      RMSE = round(arima_result$rmse, 4),
      MAPE = round(arima_result$mape, 2),
      Directional_Accuracy = round(arima_result$directional_accuracy, 2)
    ))
  }
  
  if (!is.null(rf_result)) {
    summary_data <- rbind(summary_data, data.frame(
      Model = "Random Forest",
      Stock_Code = rf_result$stock_code,
      MAE = round(rf_result$mae, 4),
      RMSE = round(rf_result$rmse, 4),
      MAPE = round(rf_result$mape, 2),
      Directional_Accuracy = round(rf_result$directional_accuracy, 2)
    ))
  }
  
  return(summary_data)
}

# Display results summary
results_summary <- create_results_summary(arima_result, rf_result)
if (nrow(results_summary) > 0) {
  print("📊 Model Performance Summary:")
  print(results_summary)
  
  # Create interactive table
  datatable(results_summary, 
            options = list(pageLength = 10, dom = 't'),
            caption = "GSE Stock Prediction Model Performance")
}

In [ ]:
# Function to make future predictions
make_future_predictions <- function(model_result, days_ahead = 5) {
  if (is.null(model_result)) return(NULL)
  
  if (model_result$model_type == "ARIMA") {
    # ARIMA future predictions
    future_forecast <- forecast(model_result$model, h = days_ahead)
    future_prices <- as.numeric(future_forecast$mean)
    confidence_intervals <- data.frame(
      lower = as.numeric(future_forecast$lower[,2]),
      upper = as.numeric(future_forecast$upper[,2])
    )
  } else if (model_result$model_type == "Random Forest") {
    # For Random Forest, we'll use the last known values to predict
    # This is a simplified approach - in practice, you'd need more sophisticated methods
    last_price <- tail(model_result$actual, 1)
    future_prices <- rep(last_price, days_ahead)  # Simplified
    confidence_intervals <- data.frame(
      lower = future_prices * 0.95,
      upper = future_prices * 1.05
    )
  }
  
  future_dates <- seq(Sys.Date() + 1, Sys.Date() + days_ahead, by = "day")
  
  future_predictions <- data.frame(
    date = future_dates,
    predicted_price = future_prices,
    lower_bound = confidence_intervals$lower,
    upper_bound = confidence_intervals$upper
  )
  
  return(future_predictions)
}

# Make future predictions
if (!is.null(arima_result)) {
  arima_future <- make_future_predictions(arima_result, 5)
  if (!is.null(arima_future)) {
    print("🔮 ARIMA Future Predictions (Next 5 Days):")
    print(arima_future)
  }
}

if (!is.null(rf_result)) {
  rf_future <- make_future_predictions(rf_result, 5)
  if (!is.null(rf_future)) {
    print("🔮 Random Forest Future Predictions (Next 5 Days):")
    print(rf_future)
  }
}

In [ ]:
# Export results to CSV
if (exists("results_summary") && nrow(results_summary) > 0) {
  write.csv(results_summary, "gse_model_results.csv", row.names = FALSE)
  cat("✓ Results exported to gse_model_results.csv\n")
}

# Export future predictions
if (exists("arima_future") && !is.null(arima_future)) {
  write.csv(arima_future, "arima_future_predictions.csv", row.names = FALSE)
  cat("✓ ARIMA future predictions exported to arima_future_predictions.csv\n")
}

if (exists("rf_future") && !is.null(rf_future)) {
  write.csv(rf_future, "rf_future_predictions.csv", row.names = FALSE)
  cat("✓ Random Forest future predictions exported to rf_future_predictions.csv\n")
}

cat("\n🎉 GSE Stock Prediction Inference Pipeline Completed Successfully!\n")
cat("📁 Check the exported CSV files for detailed results\n")